In [ ]:
import json
import random
from pathlib import Path

In [ ]:
experiment_name = "tulu3"
project_root = Path.cwd().resolve()
data_dir = project_root / "data" / "data_mixing" / experiment_name
dataset_info_path = project_root / "data" / "dataset_info.json"
target_path = data_dir / f"{experiment_name}_target.json"
sample_path = data_dir / f"{experiment_name}_sample.json"
original_path = data_dir / f"{experiment_name}_original.json"

In [ ]:
source_to_domain = {
    # Code Domain
    "ai2-adapt-dev/evol_codealpaca_heval_decontaminated": "code",
    "ai2-adapt-dev/personahub_code_v2_34999": "code",

    # Precise_IF Domain
    "ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980": "precise_IF",

    # Math Domain
    "ai2-adapt-dev/numinamath_tir_math_decontaminated": "math",
    "ai2-adapt-dev/personahub_math_v5_regen_149960": "math",
    "ai2-adapt-dev/tulu_v3.9_open_math_2_gsm8k_50k": "math",
    "ai2-adapt-dev/tulu_v3.9_personahub_math_interm_algebra_20k": "math",
    "allenai/tulu-3-sft-personas-math-grade": "math",

    # General Domain
    "ai2-adapt-dev/no_robots_converted": "general",
    "ai2-adapt-dev/oasst1_converted": "general",
    "ai2-adapt-dev/tulu_v3.9_wildchat_100k": "general",

    # Knowledge_Recall Domain
    "ai2-adapt-dev/flan_v2_converted": "knowledge_recall",
    "ai2-adapt-dev/tulu_v3.9_sciriff_10k": "knowledge_recall",
    "ai2-adapt-dev/tulu_v3.9_table_gpt_5k": "knowledge_recall",

    # Safety Domain
    "ai2-adapt-dev/coconot_converted": "safety",
    "ai2-adapt-dev/tulu_v3.9_synthetic_finalresp_wildguardmixtrain_decontaminated_50k": "safety",
    "ai2-adapt-dev/tulu_v3.9_wildjailbreak_decontaminated_50k": "safety",
}

In [ ]:
with open(target_path, "r") as read_file:
    data = json.load(read_file)

modi_data = []
for item in data:
    source = item["source"]
    item["domain"] = source_to_domain[source]
    del item["source"]
    modi_data.append(item)

with open(sample_path, "w") as write_file:
    json.dump(modi_data, write_file, indent=3)

{'instruction': 'Repeat this string "coffee in shop with flower"',
 'input': '',
 'output': 'coffee in shop with flower',
 'output_len': 6,
 'input_len': 11,
 'domain': 'general'}

In [ ]:
import sys
from typing import Any, Dict, List

def remove_keys_from_object(obj: Dict[str, Any], keys_to_remove: List[str]) -> Dict[str, Any]:
    return {k: v for k, v in obj.items() if k not in keys_to_remove}

def process_json_data(data: Any, keys_to_remove: List[str]) -> Any:
    if isinstance(data, dict):
        return remove_keys_from_object(data, keys_to_remove)
    if isinstance(data, list):
        return [remove_keys_from_object(item, keys_to_remove) if isinstance(item, dict) else item for item in data]
    raise ValueError("Unsupported JSON structure. The JSON data should be a dictionary or a list of dictionaries.")

def main(input_file: Path, output_file: Path, keys_to_remove: List[str]):
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file {input_file} does not exist.")
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        sys.exit(1)

    try:
        modified_data = process_json_data(data, keys_to_remove)
    except ValueError as e:
        print(f"Error processing JSON data: {e}")
        sys.exit(1)

    try:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(modified_data, f, ensure_ascii=False, indent=4)
    except IOError as e:
        print(f"Error writing to file {output_file}: {e}")
        sys.exit(1)

if __name__ == "__main__":
    keys_to_remove = ["output_len", "input_len", "source"]
    main(target_path, original_path, keys_to_remove)


Successfully loaded data from /mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/tulu3_target.json
Successfully removed specified keys from the JSON data.
Modified data has been saved to /mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/tulu3_original.json


In [ ]:
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

dataset_info[f"{experiment_name}_original"] = {
    "file_name": f"data_mixing/{experiment_name}/{experiment_name}_original.json"
}

for domain in ["general", "knowledge_recall", "math", "code", "safety", "precise_IF"]:
    dataset_name = f"{experiment_name}_{domain}_val"
    output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
    dataset_info[dataset_name] = {"file_name": output_path}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

In [ ]:
def load_data(file_path: Path):
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r") as read_file:
        return json.load(read_file)

def sample_tokens(data, limit, seed=42, max_passes=10):
    total_tokens = 0
    sampled_items = []
    keys_to_remove = ["output_len", "input_len", "domain"]
    pass_num = 0

    while total_tokens < limit and pass_num < max_passes:
        current_seed = seed + pass_num
        random.seed(current_seed)

        shuffled_data = data.copy()
        random.shuffle(shuffled_data)

        for item in shuffled_data:
            item_tokens = item["output_len"] + item["input_len"]
            if total_tokens + item_tokens <= limit:
                sampled_items.append(item.copy())
                total_tokens += item_tokens
            else:
                sampled_items.append(item.copy())
                total_tokens += item_tokens
                break
        else:
            pass_num += 1
            continue
        break

    if total_tokens < limit:
        print(f"Warning: Reached maximum passes ({max_passes}) without meeting token limit.")

    for item in sampled_items:
        for key in keys_to_remove:
            item.pop(key, None)

    return sampled_items

def ensure_directory_exists(directory_path: Path):
    directory_path.mkdir(parents=True, exist_ok=True)

def main():
    data_file_path = sample_path
    output_dir = data_dir
    base_token = 660_000

    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return

    token_limits = {
        "one": base_token,
        "half": base_token // 2,
        "third": base_token // 3,
        "double": base_token * 2,
        "triple": base_token * 3,
    }

    domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]

    ensure_directory_exists(output_dir)

    for domain in domains:
        domain_data = [item for item in data if item["domain"] == domain]

        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue

        for name, limit in token_limits.items():
            sampled_items = sample_tokens(domain_data, limit)
            out_filename = f"{base_token}_{domain}_{name}.json"
            out_path = output_dir / out_filename

            try:
                with open(out_path, "w") as f:
                    json.dump(sampled_items, f, indent=4)
            except IOError as e:
                print(f"Error writing to file {out_path}: {e}")
                continue

            print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")

if __name__ == "__main__":
    main()

Domain: code | Limit: 660,000 | Items: 1356
Domain: code | Limit: 330,000 | Items: 666
Domain: code | Limit: 220,000 | Items: 434
Domain: code | Limit: 1,320,000 | Items: 2734
Domain: code | Limit: 1,980,000 | Items: 4105
Domain: general | Limit: 660,000 | Items: 868
Domain: general | Limit: 330,000 | Items: 445
Domain: general | Limit: 220,000 | Items: 294
Domain: general | Limit: 1,320,000 | Items: 1730
Domain: general | Limit: 1,980,000 | Items: 2538
Domain: knowledge_recall | Limit: 660,000 | Items: 1423
Domain: knowledge_recall | Limit: 330,000 | Items: 675
Domain: knowledge_recall | Limit: 220,000 | Items: 457
Domain: knowledge_recall | Limit: 1,320,000 | Items: 2924
Domain: knowledge_recall | Limit: 1,980,000 | Items: 4417
Domain: math | Limit: 660,000 | Items: 768
Domain: math | Limit: 330,000 | Items: 371
Domain: math | Limit: 220,000 | Items: 248
Domain: math | Limit: 1,320,000 | Items: 1551
Domain: math | Limit: 1,980,000 | Items: 2337
Domain: precise_IF | Limit: 660,000 | I

In [ ]:
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
base_token = 660_000
token_limits = {
    "one": base_token,
    "half": base_token // 2,
    "third": base_token // 3,
    "double": base_token * 2,
    "triple": base_token * 3,
}

for domain in domains:
    for size in token_limits:
        dataset_name = f"{base_token}_{domain}_{size}"
        output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
        dataset_info[dataset_name] = {"file_name": output_path}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

## Equal weights for Tulu3

In [ ]:


equal_tokens = int(462_841_566 / 6)
domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]

try:
    data = load_data(sample_path)
except (FileNotFoundError, json.JSONDecodeError) as e:
    print(f"Error loading data: {e}")
else:
    ensure_directory_exists(data_dir)

    for domain in domains:
        domain_data = [item for item in data if item["domain"] == domain]

        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue

        sampled_items = sample_tokens(domain_data, equal_tokens)
        out_path = data_dir / f"{domain}_equal.json"

        try:
            with open(out_path, "w") as f:
                json.dump(sampled_items, f, indent=4)
        except IOError as io_error:
            print(f"Error writing to file {out_path}: {io_error}")
            continue

        print(f"Domain: {domain} | Limit: {equal_tokens:,} | Items: {len(sampled_items)}")

Domain: code | Limit: 77,140,261 | Items: 158950
Domain: general | Limit: 77,140,261 | Items: 103006
Domain: knowledge_recall | Limit: 77,140,261 | Items: 174547
Domain: math | Limit: 77,140,261 | Items: 89279
Domain: precise_IF | Limit: 77,140,261 | Items: 207668
Domain: safety | Limit: 77,140,261 | Items: 359488


In [ ]:
domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
data_files = [data_dir / f"{domain}_equal.json" for domain in domains]

combined_data = []
for data_source in data_files:
    with open(data_source, "r") as read_file:
        combined_data.extend(json.load(read_file))

dataset_name = f"{experiment_name}_equal"
with open(data_dir / f"{dataset_name}.json", "w") as f:
    json.dump(combined_data, f, indent=2)

In [ ]:
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
for domain in domains:
    dataset_name = f"{domain}_equal"
    output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
    dataset_info[dataset_name] = {"file_name": output_path}

dataset_name = f"{experiment_name}_equal"
dataset_info[dataset_name] = {
    "file_name": f"data_mixing/{experiment_name}/{dataset_name}.json"
}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

## Ours

In [ ]:
domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
ratios = [0.19264953, 0.16925506, 0.12808657, 0.10103714, 0.15628441, 0.25268729]
total_tokens = 462_841_566

if len(domains) != len(ratios):
    raise ValueError("The number of ratios must match the number of domains.")

token_limits = {domain: int(total_tokens * ratio) for domain, ratio in zip(domains, ratios)}

try:
    data = load_data(sample_path)
except (FileNotFoundError, json.JSONDecodeError) as e:
    print(f"Error loading data: {e}")
else:
    ensure_directory_exists(data_dir)
    all_sampled_items = []

    for domain in domains:
        domain_data = [item for item in data if item.get("domain") == domain]

        if not domain_data:
            print(f"Warning: No data found for domain '{domain}'. Skipping.")
            continue

        limit = token_limits[domain]
        sampled_items = sample_tokens(domain_data, limit)
        all_sampled_items.extend(sampled_items)

        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")

    combined_output_file = data_dir / f"{experiment_name}_ours.json"

    try:
        with open(combined_output_file, "w") as f:
            json.dump(all_sampled_items, f, indent=4)
        print(f"\nAll sampled data has been saved to '{combined_output_file}'.")
    except IOError as io_error:
        print(f"Error writing to file {combined_output_file}: {io_error}")

Domain: code | Limit: 89,166,210 | Items: 183782
Domain: general | Limit: 78,338,277 | Items: 104622
Domain: knowledge_recall | Limit: 59,283,788 | Items: 134182
Domain: math | Limit: 46,764,188 | Items: 54247
Domain: precise_IF | Limit: 72,334,921 | Items: 194741
Domain: safety | Limit: 116,954,181 | Items: 545041

All sampled data has been saved to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/tulu3_ours.json'.


In [ ]:
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

dataset_name = f"{experiment_name}_ours"
dataset_info[dataset_name] = {
    "file_name": f"data_mixing/{experiment_name}/{dataset_name}.json"
}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

## Ours Tulu3

In [ ]:
domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
ratios = [0.16863725, 0.12637526, 0.18440989, 0.18205498, 0.13835368, 0.20016894]
total_tokens = 462_841_566

if len(domains) != len(ratios):
    raise ValueError("The number of ratios must match the number of domains.")

token_limits = {domain: int(total_tokens * ratio) for domain, ratio in zip(domains, ratios)}

try:
    data = load_data(sample_path)
except (FileNotFoundError, json.JSONDecodeError) as e:
    print(f"Error loading data: {e}")
else:
    ensure_directory_exists(data_dir)
    all_sampled_items = []

    for domain in domains:
        domain_data = [item for item in data if item.get("domain") == domain]

        if not domain_data:
            print(f"Warning: No data found for domain '{domain}'. Skipping.")
            continue

        limit = token_limits[domain]
        sampled_items = sample_tokens(domain_data, limit)
        all_sampled_items.extend(sampled_items)

        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")

    combined_output_file = data_dir / f"{experiment_name}_Qwen_ours.json"

    try:
        with open(combined_output_file, "w") as f:
            json.dump(all_sampled_items, f, indent=4)
        print(f"\nAll sampled data has been saved to '{combined_output_file}'.")
    except IOError as io_error:
        print(f"Error writing to file {combined_output_file}: {io_error}")

Domain: code | Limit: 78,052,328 | Items: 160841
Domain: general | Limit: 58,491,723 | Items: 78048
Domain: knowledge_recall | Limit: 85,352,562 | Items: 192998
Domain: math | Limit: 84,262,612 | Items: 97482
Domain: precise_IF | Limit: 64,035,833 | Items: 172444
Domain: safety | Limit: 92,646,505 | Items: 431679

All sampled data has been saved to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/tulu3_Qwen_ours.json'.


In [ ]:
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

dataset_name = f"{experiment_name}_Qwen_ours"
dataset_info[dataset_name] = {
    "file_name": f"data_mixing/{experiment_name}/{dataset_name}.json"
}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)